In [6]:
import pickle
import pandas as pd
import numpy as np
from kramersmoyal import km
import matplotlib.pyplot as plt
from utils.Data_cleaning import data_cleaning
from utils.Functions import data_filter, integrate_omega, KM_Coeff_1, KM_Coeff_2, daily_profile, power_mismatch, exp_decay, Euler_Maruyama, Increments, autocor
import os

In [7]:
# ---- GRIDS DEFINITION ----

grids = ['SK']


delta_t_map = {i: 1.0 for i in grids}  # store synth sampling per grid
data_orig = {i: [] for i in grids}
increments_orig = {i: [] for i in grids}
autocor_orig = {i: [] for i in grids}

In [8]:
# ---- SYNTHETIC DATASET CREATION ----

synth_data_model_2 = {i:[]for i in grids}
increments_model_2 = {i:[]for i in grids}
autocor_model_2 = {i:[]for i in grids}
c_1_model2 = {i:[]for i in grids}
epsilon_model2 = {i:[]for i in grids}

edges_1d     = {i:[]for i in grids}
drift_1d     = {i:[]for i in grids}
diffusion_1d = {i:[]for i in grids}
edges_2d     = {i:[]for i in grids}
kmc_2d       = {i:[]for i in grids}

# ---- PARAMETER ESTIMATION FOR THE SK GRID ----

for grid in grids:

    pkl_path = os.path.join('dataset', f'Frequency_data_{grid}.pkl')

    try:
        raw = pd.read_pickle(pkl_path)
    except Exception:
        with open(pkl_path, 'rb') as f:
            raw = pickle.load(f)

    print("First 2 lines of the South Korean power grid frequency dataset:")
    print(raw.head(2))
    print(f"\nDataset shape: {raw.shape}")
    print(f"Column names: {list(raw.columns)}")
    print(f"\nExact number of lines: {len(raw):,}")
    print(f"Number of rows: {raw.shape[0]:,}")
    print(f"Number of columns: {raw.shape[1]}")
    print(f"\nData types:")
    print(raw.dtypes)
    print(f"\nTime range:")
    print(f"Start: {raw.index.min()}")
    print(f"End: {raw.index.max()}")
    print(f"Duration: {raw.index.max() - raw.index.min()}")
    print(f"\nMissing values:")
    print(raw.isnull().sum())

    print('###' * 50)

    freq = raw['freq'].squeeze()

    print(freq.head(2))

    freq = data_cleaning(freq)

    data_orig[grid] = np.asarray(freq)  # original frequency series (Hz)
    increments_orig[grid] = Increments(data_orig[grid], time_res=1, step=1)
    autocor_orig[grid] = autocor(data_orig[grid], steps=1, time_res=1)
    data = (freq-60)*(2*np.pi)   #Use the angular velocity for the calcualations



    trend = 1
    bw_drift = 0.1
    bw_diff = 0.1
    dist_drift = 500    #for large data set: dist_drift = 350 for Balearic
    dist_diff = 500


    if grid == 'SK':
      Delta_P = power_mismatch(data,avg_for_each_hour = False,dispatch=2,start_minute=0,end_minute=1/6,length_seconds_of_interval=5)
      dispatch = 2


    data_for_km = data - trend*data_filter(data)

    data_for_km.head(10)


First 2 lines of the South Korean power grid frequency dataset:
                          freq QI
2024-08-15 00:00:00  60.021490  0
2024-08-15 00:00:01  60.025718  0

Dataset shape: (10108800, 2)
Column names: ['freq', 'QI']

Exact number of lines: 10,108,800
Number of rows: 10,108,800
Number of columns: 2

Data types:
freq    float64
QI       object
dtype: object

Time range:
Start: 2024-08-15 00:00:00
End: 2024-12-09 23:59:59
Duration: 116 days 23:59:59

Missing values:
freq    557296
QI           0
dtype: int64
######################################################################################################################################################
2024-08-15 00:00:00    60.021490
2024-08-15 00:00:01    60.025718
Freq: s, Name: freq, dtype: float64
Starting data cleaning ____________________
Number of too high frequency values:  0 Number of too low frequency values:  0
Number of too large increments:  36
Number of windows with constant frequency for longer than 60s:  0
Nu

/pfs/data6/home/ka/ka_iai/ka_bj2362/dsr/utils/Data_cleaning.py:49: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_cl = data_m.fillna(method='ffill', limit=N_f)


In [9]:

    c_1 = KM_Coeff_1(data_for_km,dim= 1,time_res = 1,bandwidth = bw_drift,dist = dist_drift, order = 1)
    c_2_decay = trend*exp_decay(data,time_res=1,size = 899)
    epsilon = KM_Coeff_2(data_for_km,dim = 1,time_res = 1,bandwidth = bw_diff,dist = dist_diff,multiplicative_noise = False)

    kmc,edges = km(data_for_km,powers = [0,1,2],bins = np.array([6000]),bw=bw_drift)
    edges_1d[grid] = edges[0]
    drift_1d[grid] = kmc[1]
    diffusion_1d[grid] = kmc[2]
    c_1_model2[grid] = c_1
    epsilon_model2[grid] = epsilon

    delta_t = 0.1  # time step for Euler-Maruyama
    delta_t_map[grid] = delta_t  # store delta_t used for synthesis

    # Omega is the angular frequency deviation from nominal value (in rad/s).
    omega_synth_model_2 = Euler_Maruyama(data,c_1=c_1,c_2_decay=c_2_decay,Delta_P = Delta_P,epsilon=epsilon,time_res = 1,dispatch = dispatch,delta_t=delta_t,t_final=5,model=2,factor_daily_profile=0)
    # Convert back to frequency in Hz
    freq_synth_model_2 = omega_synth_model_2/(2*np.pi) + 60

    synth_data_model_2[grid].append(freq_synth_model_2)
    increments_model_2[grid].append(Increments(freq_synth_model_2,time_res = delta_t,step = 1))
    autocor_model_2[grid].append(autocor(freq_synth_model_2,steps = int(1/delta_t),time_res = delta_t))

ValueError: array must not contain infs or NaNs